In [1]:
# Aggregate dataset to Plant,Material,Channel level
# Reuse the clean, processed file created for the lowest granularity - Plant,Material,Channel,DPGrp

import os
import sys
import time
import pandas as pd
import numpy as np

pd.set_option("display.max_colwidth", 2000)

/home/rahul/anaconda3/envs/fmldk/lib/python3.11/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/rahul/anaconda3/envs/fmldk/lib/python3.11/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


In [2]:
cleaned_input_filepath = "../data/basic_input.xlsx" # copied from the Plant-Material-Channel-DPGrp
aggregated_output_filepath = "../data/plant_matgrp_agg_input.xlsx"

##### Aggregate keys from Plant-Material-DC-Customer to Plant-Material-DC level

In [12]:
df = pd.read_excel(cleaned_input_filepath)

In [13]:
df.head()

,Plant,Material,Material_Desc,Channel,DP_Group_L3,DP_Group_L3_Desc,YearMonth,Customer Demand Qty,Customer Demand Value,Material_Group,Material_Group_Desc,Material_Hierarchy,Material_Hierarchy_Desc,Valuation_Class,Valuation_Class_Desc,Minimum Lot Size,Working_Days,key
0,3891,0204802919,Change Lever Unit,10,D13,OE_FAW,2020-01-01,160.0,9619.12,PG016,Transmission,16EE,Change Lever Unit,3100,Trading goods,80,16,3891_0204802919_10_D13
1,3891,0204802919,Change Lever Unit,10,D13,OE_FAW,2020-02-01,240.0,14428.68,PG016,Transmission,16EE,Change Lever Unit,3100,Trading goods,80,20,3891_0204802919_10_D13
2,3891,0204802919,Change Lever Unit,10,D13,OE_FAW,2020-03-01,720.0,43286.05,PG016,Transmission,16EE,Change Lever Unit,3100,Trading goods,80,22,3891_0204802919_10_D13
3,3891,0204802919,Change Lever Unit,10,D13,OE_FAW,2020-04-01,560.0,33666.93,PG016,Transmission,16EE,Change Lever Unit,3100,Trading goods,80,21,3891_0204802919_10_D13
4,3891,0204802919,Change Lever Unit,10,D13,OE_FAW,2020-05-01,480.0,28857.37,PG016,Transmission,16EE,Change Lever Unit,3100,Trading goods,80,18,3891_0204802919_10_D13


In [14]:
df.dtypes

Plant                               int64
Material                              str
Material_Desc                         str
Channel                             int64
DP_Group_L3                           str
DP_Group_L3_Desc                      str
YearMonth                  datetime64[us]
Customer Demand Qty               float64
Customer Demand Value             float64
Material_Group                        str
Material_Group_Desc                   str
Material_Hierarchy                    str
Material_Hierarchy_Desc               str
Valuation_Class                     int64
Valuation_Class_Desc                  str
Minimum Lot Size                    int64
Working_Days                        int64
key                                   str
dtype: object

In [15]:
# Verify uniqueness of attributes at key level
KEY = ["Plant", "Material_Group", "Channel"]
DROP_COLS = ["Material", "Material_Desc", 'Material_Hierarchy', 'Material_Hierarchy_Desc', 'Valuation_Class', 'Valuation_Class_Desc', 'Minimum Lot Size', "DP_Group_L3", "DP_Group_L3_Desc", "key"]
MEASURES = ["Customer Demand Qty", "Customer Demand Value"]
TIME = ["YearMonth"]

# Working_Days: treat as "time-varying attribute" to be averaged per KEY+TIME
TIME_VARYING_ATTRS = ["Working_Days"]  # extendable list if needed later

df_chk = df.drop(columns=[c for c in DROP_COLS if c in df.columns]).copy()

attr_cols = [c for c in df_chk.columns if c not in KEY + TIME + MEASURES]

violations = {}
for c in attr_cols:
    nun = df_chk.groupby(KEY)[c].nunique(dropna=True)
    bad = nun[nun > 1]
    if not bad.empty:
        violations[c] = bad

len(violations), list(violations.keys())

(1, ['Working_Days'])

In [17]:


def aggregate_fast_validated(df: pd.DataFrame) -> pd.DataFrame:
    # 1) Drop unwanted columns
    df = df.drop(columns=[c for c in DROP_COLS if c in df.columns]).copy()

    # 2) Ensure datetime
    df["YearMonth"] = pd.to_datetime(df["YearMonth"])

    group_cols = KEY + TIME

    # 3) Identify candidate attribute columns (everything except keys/time/measures)
    attr_cols = [c for c in df.columns if c not in group_cols + MEASURES]

    # Split attributes into:
    # - static per KEY (validate uniqueness per KEY)
    # - time-varying per KEY+TIME (aggregate per KEY+TIME)
    time_varying = [c for c in TIME_VARYING_ATTRS if c in df.columns]
    static_attrs = [c for c in attr_cols if c not in time_varying]

    # A) FAST UNIQUENESS VALIDATION for STATIC attributes at KEY level
    if static_attrs:
        nun = df.groupby(KEY, dropna=False)[static_attrs].nunique(dropna=True)
        violating_cols = nun.columns[(nun > 1).any()].tolist()

        if violating_cols:
            diag = {}
            for c in violating_cols[:10]:
                bad_keys = nun.index[nun[c] > 1].tolist()[:5]
                diag[c] = bad_keys

            raise ValueError(
                f"Static attribute(s) not unique per key {KEY}. "
                f"Violating columns: {violating_cols}\n"
                f"Sample offending keys (first 5): {diag}\n"
                f"Note: '{time_varying}' are excluded from static validation."
            )

        # Extract static attributes once per KEY (safe because validated)
        df_static = df.groupby(KEY, as_index=False, dropna=False)[static_attrs].first()
    else:
        df_static = df[KEY].drop_duplicates()

    # B) Aggregate measures + time-varying attributes at KEY+TIME level
    agg_dict = {m: "sum" for m in MEASURES}
    # Working_Days mean per month per key
    for c in time_varying:
        agg_dict[c] = "mean"

    df_km = (
        df.groupby(group_cols, as_index=False, dropna=False)
          .agg(agg_dict)
    )

    # C) Join static attrs back onto monthly fact table
    out = df_km.merge(df_static, on=KEY, how="left")

    # Optional: cast Working_Days to int
    if "Working_Days" in out.columns:
        # If mean resulted in 22.0 etc., safe to round and convert
        # Comment out if you want float
        out["Working_Days"] = out["Working_Days"].round().astype("Int64")

    # Sanity: enforce unique grain
    dup = out.duplicated(group_cols).sum()
    if dup:
        raise AssertionError(f"Unexpected duplicates after aggregation at {group_cols}: {dup}")

    return out

df_agg = aggregate_fast_validated(df)


In [18]:
df_agg.shape

(2976, 8)

In [19]:
df_agg.head()

,Plant,Material_Group,Channel,YearMonth,Customer Demand Qty,Customer Demand Value,Working_Days,Material_Group_Desc
0,3891,PG011,10,2020-01-01,65060.0,8492467.20,16,Compressors
1,3891,PG011,10,2020-02-01,49700.0,6390045.92,20,Compressors
2,3891,PG011,10,2020-03-01,71729.0,9507593.91,22,Compressors
3,3891,PG011,10,2020-04-01,76710.0,9757240.70,21,Compressors
4,3891,PG011,10,2020-05-01,79624.0,10250335.63,18,Compressors


In [20]:
# Verify Each (Plant, Material, Channel, YearMonth) should appear once

assert df_agg.duplicated(["Plant","Material_Group","Channel","YearMonth"]).sum() == 0

In [21]:
# create key col
df_agg['key'] = df_agg['Plant'].astype(str) + \
'_' + df_agg['Material_Group'].astype(str) + \
'_' + df_agg['Channel'].astype(str) 

# check nulls (shouldn't be present in categorical or datetime columns)
print(df_agg.isnull().any())

# write out
df_agg.to_excel(aggregated_output_filepath, index=False)

Plant                    False
Material_Group           False
Channel                  False
YearMonth                False
Customer Demand Qty      False
Customer Demand Value    False
Working_Days             False
Material_Group_Desc      False
key                      False
dtype: bool
